In [13]:
from pyspark.sql import SparkSession   # to create Spark environment
from pyspark.sql.functions import col  # for dataframe column operations
from pyspark.ml.recommendation import ALS   # Alternating Least Squares collaborative filtering model
from pyspark.ml.evaluation import RegressionEvaluator   # to evaluate recommendation model

## 1. Load Dataset and Create Spark Session

In [14]:
spark = SparkSession.builder.appName("MovieLensRecommender").getOrCreate()  # start spark session

In [15]:
from pyspark.sql import SparkSession   # create Spark environment
from pyspark.sql.functions import col  # column operations

# start Spark session
spark = SparkSession.builder.appName("MovieLens_Pearson_CF").getOrCreate()

# load MovieLens ratings dataset
ratings = spark.read.csv(
    "/home/rajesh/CSL7100/Assignment3/ml-latest-small/ratings.csv",
    header=True,
    inferSchema=True
)

ratings.show(5)  # preview data

26/03/14 22:11:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows



In [24]:
print((ratings.count(), len(ratings.columns)))


(100836, 4)


## 2. Find Common Movies Rated by User Pairs

In [16]:
ratings_u1 = ratings.alias("r1")  # alias for first user
ratings_u2 = ratings.alias("r2")  # alias for second user

# join ratings where users rated the same movie
user_pairs = ratings_u1.join(
    ratings_u2,
    (col("r1.movieId") == col("r2.movieId")) & (col("r1.userId") < col("r2.userId"))
)

user_pairs.select(
    col("r1.userId").alias("user1"),
    col("r2.userId").alias("user2"),
    col("r1.movieId"),
    col("r1.rating").alias("rating1"),
    col("r2.rating").alias("rating2")
).show(5)

+-----+-----+-------+-------+-------+
|user1|user2|movieId|rating1|rating2|
+-----+-----+-------+-------+-------+
|    1|  610|      1|    4.0|    5.0|
|    1|  609|      1|    4.0|    3.0|
|    1|  608|      1|    4.0|    2.5|
|    1|  607|      1|    4.0|    4.0|
|    1|  606|      1|    4.0|    2.5|
+-----+-----+-------+-------+-------+
only showing top 5 rows



In [25]:
print((user_pairs.count(), len(user_pairs.columns)))

(2911932, 8)


## 3. Compute Pearson Similarity

In [17]:
from pyspark.sql.functions import corr  # Pearson correlation function

# compute Pearson similarity between user pairs
user_similarity = user_pairs.groupBy(
    col("r1.userId").alias("user1"),
    col("r2.userId").alias("user2")
).agg(
    corr("r1.rating", "r2.rating").alias("pearson_similarity")
)

user_similarity.show(10)

+-----+-----+--------------------+
|user1|user2|  pearson_similarity|
+-----+-----+--------------------+
|    1|  587| -0.3021951880134726|
|    2|  582| 0.04419417382415873|
|    4|  599| 0.08060613049558178|
|    7|  475|  0.2790802022945441|
|    9|  564|                null|
|   12|  589|                null|
|   16|  584|-3.63405789187685...|
|   18|  552|  0.5760499929667433|
|   21|  600| -0.2271096581618072|
|   26|  594|-4.70356279334185...|
+-----+-----+--------------------+
only showing top 10 rows



In [26]:
print((user_similarity.count(), len(user_similarity.columns)))

(164054, 3)


## 4. Predict ratings for a given user using weighted average of neighbors

## Find Most Similar Users

In [19]:
target_user = 2

similar_users = user_similarity.filter(
    (col("user1") == target_user) | (col("user2") == target_user)
).orderBy(col("pearson_similarity").desc())

similar_users.show(5)

+-----+-----+------------------+
|user1|user2|pearson_similarity|
+-----+-----+------------------+
|    2|  363|               1.0|
|    2|   60|               1.0|
|    2|   34|               1.0|
|    2|  148|               1.0|
|    2|  240|               1.0|
+-----+-----+------------------+
only showing top 5 rows



In [27]:
print((similar_users.count(), len(similar_users.columns)))

(451, 3)


## collect Movies Rated by Similar Users

In [20]:
top_users = similar_users.limit(5)  # select top 5 similar users

similar_user_ids = top_users.select("user2").rdd.flatMap(lambda x: x).collect()

# get ratings from those users
similar_user_ratings = ratings.filter(col("userId").isin(similar_user_ids))

similar_user_ratings.show(5)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|    33|      1|   3.0|939647444|
|    33|      7|   1.0|939654896|
|    33|     11|   2.0|939654896|
|    33|     17|   4.0|939654828|
|    33|     21|   4.0|939715904|
+------+-------+------+---------+
only showing top 5 rows



In [28]:
print((similar_user_ratings.count(), len(similar_user_ratings.columns)))

(448, 4)


## Generate Movie Recommendations

In [21]:
from pyspark.sql.functions import avg

recommended_movies = similar_user_ratings.groupBy("movieId") \
    .agg(avg("rating").alias("predicted_rating")) \
    .orderBy(col("predicted_rating").desc())

recommended_movies.show(10)

+-------+----------------+
|movieId|predicted_rating|
+-------+----------------+
|    673|             5.0|
|    368|             5.0|
|     28|             5.0|
|    516|             5.0|
|   2355|             5.0|
|    613|             5.0|
|    914|             5.0|
|    593|             5.0|
|   1210|             5.0|
|    253|             5.0|
+-------+----------------+
only showing top 10 rows



In [29]:
print((recommended_movies.count(), len(recommended_movies.columns)))

(406, 2)


In [22]:
watched_movies = ratings.filter(col("userId") == target_user).select("movieId")

final_recommendations = recommended_movies.join(
    watched_movies,
    on="movieId",
    how="left_anti"   # removes movies already watched
)

final_recommendations.show(5)

+-------+----------------+
|movieId|predicted_rating|
+-------+----------------+
|    613|             5.0|
|   1653|             5.0|
|    673|             5.0|
|   2580|             5.0|
|    593|             5.0|
+-------+----------------+
only showing top 5 rows



In [30]:
print((watched_movies.count(), len(watched_movies.columns)))

(29, 1)


In [31]:
print((final_recommendations.count(), len(final_recommendations.columns)))

(400, 2)


In [35]:
from pyspark.sql.functions import col

predictions_with_actual = predictions.join(
    ratings.select("userId","movieId","rating"),   # actual ratings
    on=["userId","movieId"],                       # join keys
    how="inner"
)

predictions_with_actual.show(5)

+------+-------+----------------+------+
|userId|movieId|predicted_rating|rating|
+------+-------+----------------+------+
|    33|   1271|             3.0|   3.0|
|    33|    223|             4.0|   4.0|
|    33|   1188|             4.0|   4.0|
|    33|   1641|             5.0|   5.0|
|   363|   1246|             4.0|   4.0|
+------+-------+----------------+------+
only showing top 5 rows



In [36]:
from pyspark.sql.functions import pow, avg, sqrt

rmse_df = predictions_with_actual.withColumn(
    "squared_error",
    pow(col("rating") - col("predicted_rating"), 2)  # squared error
)

rmse = rmse_df.select(
    sqrt(avg("squared_error")).alias("RMSE")
)

rmse.show()

+----+
|RMSE|
+----+
| 0.0|
+----+



In [12]:
spark.stop()